In [ ]:
# Configuracion de cluster Slurm (alineada con run.sh / run_parallel.sh)
SLURM_PARTITION = 'long'
SLURM_CPUS_PER_TASK = 8
SLURM_MEM = '32G'
SLURM_GRES_EXPERIMENT = 'shard:4'
SLURM_GRES_MASSIVE = 'shard:6'
CONDA_ENV = 'RFA2526pt'
PYTORCH_CUDA_ALLOC_CONF = 'expandable_segments:True'

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = PYTORCH_CUDA_ALLOC_CONF

print('Partition:', SLURM_PARTITION)
print('CPUs:', SLURM_CPUS_PER_TASK, '| Mem:', SLURM_MEM)
print('GRES exp:', SLURM_GRES_EXPERIMENT, '| GRES masivo:', SLURM_GRES_MASSIVE)
print('Conda env:', CONDA_ENV)
print('PYTORCH_CUDA_ALLOC_CONF:', os.environ['PYTORCH_CUDA_ALLOC_CONF'])


# 08 - Fusion Avanzada (Late Fusion)

Este notebook combina las tecnicas de limpieza avanzadas:
- Audio: VAD + Wav2Vec2.
- Video: limpieza temporal + CLIP.
- Sensorial: filtrado IQR + ventanas alineadas a texto.
- Texto: embeddings de xlm-roberta-base.

Estrategia de fusion tardia:
1. Se entrena un clasificador por modalidad.
2. Se promedian probabilidades de las modalidades.
3. Se exportan tres JSON (ES, EN, ES+EN) sobre el conjunto completo disponible.


In [ ]:
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import torch
from tqdm.auto import tqdm

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from transformers import AutoTokenizer, AutoModel

DATA_JSON_PATH = Path('/home/alumno.upv.es/scheng1/EXIST 2026 Videos Dataset/training/EXIST2026_training.json')
PROJECT_ROOT = Path('/home/alumno.upv.es/scheng1/Master-IA-ALC/lab3/notebooks_shiyi')
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
DELIVERABLES_DIR = PROJECT_ROOT / 'entregables'
TEXT_CACHE_NPZ = ARTIFACTS_DIR / 'text_xlmr_features.npz'

AUDIO_NPZ = ARTIFACTS_DIR / 'audio_vad_w2v_features.npz'
VIDEO_NPZ = ARTIFACTS_DIR / 'video_clean_clip_features.npz'
SENSOR_NPZ = ARTIFACTS_DIR / 'sensorial_filtered_features.npz'

for p in [AUDIO_NPZ, VIDEO_NPZ, SENSOR_NPZ]:
    if not p.exists():
        raise FileNotFoundError(f'Missing artifact: {p}. Run notebooks 05/06/07 first.')

DELIVERABLES_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


In [ ]:
with open(DATA_JSON_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)

rows = []
for sid, obj in raw.items():
    rec = {'sample_id': str(sid)}
    rec.update(obj)
    rows.append(rec)

df = pd.DataFrame(rows)

def majority_task3_1(lbls):
    vals = [x for x in lbls if x in {'YES', 'NO'}]
    if not vals:
        return 'UNKNOWN'
    c = Counter(vals)
    if c['YES'] == c['NO']:
        return vals[0]
    return c.most_common(1)[0][0]

df['label'] = df['labels_task3_1'].apply(majority_task3_1)
df['y'] = df['label'].map({'NO': 0, 'YES': 1}).fillna(-1).astype(int)
df = df.sort_values('sample_id').reset_index(drop=True)

sample_ids = df['sample_id'].astype(str).to_numpy()
y = df['y'].to_numpy(np.int64)
langs = df['lang'].astype(str).to_numpy()
texts = df['text'].fillna('').astype(str).tolist()


In [ ]:
# Cargar modalidades ya limpias (audio/video/sensorial) y alinearlas por sample_id.
def load_modality(npz_path, x_key):
    data = np.load(npz_path, allow_pickle=True)
    X = data[x_key]
    sid = data['sample_ids'].astype(str)
    idx = {s: i for i, s in enumerate(sid)}
    pos = [idx[s] for s in sample_ids]
    return X[pos]

X_audio = load_modality(AUDIO_NPZ, 'X_audio')
X_video = load_modality(VIDEO_NPZ, 'X_video')
X_sensor = load_modality(SENSOR_NPZ, 'X_sensor')

print('Audio shape:', X_audio.shape)
print('Video shape:', X_video.shape)
print('Sensor shape:', X_sensor.shape)


In [ ]:
# Embeddings de texto XLM-R para incluir modalidad textual en fusion tardia.
MODEL_NAME = 'xlm-roberta-base'
MAX_LEN = 256
BATCH = 16

if TEXT_CACHE_NPZ.exists():
    c = np.load(TEXT_CACHE_NPZ, allow_pickle=True)
    X_text = c['X_text']
    sid = c['sample_ids'].astype(str)
    idx = {s: i for i, s in enumerate(sid)}
    X_text = X_text[[idx[s] for s in sample_ids]]
    print('Loaded text cache:', X_text.shape)
else:
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    mdl = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
    mdl.eval()

    embs = []
    for i in tqdm(range(0, len(texts), BATCH), desc='XLM-R text embeddings'):
        bt = texts[i:i + BATCH]
        enc = tok(bt, padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt')
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            h = mdl(**enc).last_hidden_state[:, 0, :]
        embs.append(h.cpu().numpy())

    X_text = np.vstack(embs).astype(np.float32)
    np.savez_compressed(TEXT_CACHE_NPZ, X_text=X_text, sample_ids=sample_ids)
    print('Saved text cache:', X_text.shape)


In [ ]:
# Fusion tardia: un clasificador por modalidad y promedio de probabilidades.
def fit_modality_clf(Xtr, ytr):
    clf = Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(max_iter=2500, class_weight='balanced', n_jobs=-1)),
    ])
    clf.fit(Xtr, ytr)
    return clf


def late_fusion_predict(config_langs, exp_tag):
    m = np.isin(langs, config_langs) & (y >= 0)
    ytr = y[m]

    clfs = {
        'audio': fit_modality_clf(X_audio[m], ytr),
        'video': fit_modality_clf(X_video[m], ytr),
        'sensor': fit_modality_clf(X_sensor[m], ytr),
        'text': fit_modality_clf(X_text[m], ytr),
    }

    probs = []
    probs.append(clfs['audio'].predict_proba(X_audio)[:, 1])
    probs.append(clfs['video'].predict_proba(X_video)[:, 1])
    probs.append(clfs['sensor'].predict_proba(X_sensor)[:, 1])
    probs.append(clfs['text'].predict_proba(X_text)[:, 1])

    p_mean = np.mean(np.vstack(probs), axis=0)
    pred = np.where(p_mean >= 0.5, 'YES', 'NO')

    out = [
        {'test_case': 'EXIST2026', 'id': str(sid), 'value': str(lbl)}
        for sid, lbl in zip(sample_ids, pred)
    ]
    out_path = DELIVERABLES_DIR / f'BeingChillingWeWillWin_{exp_tag}.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(out, f, ensure_ascii=False)
    print('Saved', out_path, '| rows=', len(out))


late_fusion_predict(['es'], '08FusionLate_ES')
late_fusion_predict(['en'], '08FusionLate_EN')
late_fusion_predict(['es', 'en'], '08FusionLate_ES_EN')
